In [ ]:
# %% 1. Configuración
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import trackpy as tp
import scienceplots                       # registra los estilos
import warnings
from scipy.signal import savgol_filter

tp.quiet()
warnings.filterwarnings('ignore', category=UserWarning, module='trackpy')
warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.style.use(['science', 'no-latex'])
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110,
                     'axes.grid': True, 'grid.alpha': 0.3})

VIDEO = "muestraperdida.avi"
T_INICIO_S = None            # recorte temporal en segundos; None = desde el inicio
T_FIN_S = 20                 # None = hasta el final

# Calibración y constantes de la muestra.
RADIO_UM = 0.80              # radio de la esfera atrapada, del cociente de difusiones
RADIO_UM_ERR = 0.14
ESCALA_UM_PX = 0.0946        # calibración con retículo de paso conocido
ESCALA_UM_PX_ERR = 0.002
ETA_MEDIDA = 2.36            # mPa s, del análisis browniano
ETA_MEDIDA_ERR = 0.30

CANAL = 'verde'              # 'verde' | 'rojo' | 'azul' | 'gris'
ROI = None                   # (y0, y1, x0, x1); None = automático
POSICION_TRAMPA = None       # (x, y); None = máximo de rojo menos verde

SALIDA = "figuras_pinza"
FORMATO = 'svg'
DPI = 300
os.makedirs(SALIDA, exist_ok=True)

try:
    import numba                                              # noqa: F401
    ENGINE = 'numba'
except ImportError:
    ENGINE = 'auto'


def guardar(nombre):
    ruta = os.path.join(SALIDA, f"{nombre}.{FORMATO}")
    plt.savefig(ruta, dpi=DPI, bbox_inches='tight')
    return ruta


In [ ]:
# %% 2. Video y ventana temporal
cap = cv2.VideoCapture(VIDEO)
if not cap.isOpened():
    raise IOError(f"No se pudo abrir {VIDEO}")
FPS = cap.get(cv2.CAP_PROP_FPS)
N_TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
DT = 1.0/FPS
print(f"{VIDEO}: {N_TOTAL} frames, {FPS:.3f} fps, {N_TOTAL/FPS:.1f} s")

# La ventana sirve para descartar los tramos arruinados (la cámara se movió, la
# partícula se escapó antes de tiempo). El tiempo se cuenta desde su primer
# frame y todo lo que sigue trabaja adentro.
K_INI = 0 if T_INICIO_S is None else max(int(round(T_INICIO_S*FPS)), 0)
K_FIN = (N_TOTAL - 1 if T_FIN_S is None
         else min(int(round(T_FIN_S*FPS)), N_TOTAL - 1))
if K_FIN - K_INI < 10:
    raise ValueError("Ventana temporal demasiado corta.")
print(f"ventana: frames {K_INI}-{K_FIN}, {(K_FIN - K_INI + 1)/FPS:.1f} s")


def extraer_canal(frame, canal=CANAL):
    if frame.ndim == 2:
        return frame.astype(np.float32)
    i = {'azul': 0, 'verde': 1, 'rojo': 2}.get(canal)
    g = frame[:, :, i] if i is not None else cv2.cvtColor(frame,
                                                          cv2.COLOR_BGR2GRAY)
    return g.astype(np.float32)


def un_frame_color(k):
    cap = cv2.VideoCapture(VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(k))
    ok, fr = cap.read()
    cap.release()
    if not ok:
        raise IOError(f"No se pudo leer el frame {k}")
    return fr


In [ ]:
# %% 3. Región de interes y posición de la trampa
# El ROI recorta los bordes del encuadre: los filtros posteriores son
# convoluciones y un borde abrupto genera una respuesta propia.
frame_crudo = un_frame_color(K_INI)
img0 = extraer_canal(frame_crudo)
if ROI is None:
    fy = np.where(img0.mean(axis=1) > 30)[0]
    fx = np.where(img0.mean(axis=0) > 30)[0]
    ROI = (int(fy.min()) + 5, int(fy.max()) - 5,
           int(fx.min()) + 5, int(fx.max()) - 5)
print(f"ROI {ROI} -> {ROI[1]-ROI[0]}x{ROI[3]-ROI[2]} px")

rec_color = frame_crudo[ROI[0]:ROI[1], ROI[2]:ROI[3]]

# El láser es rojo: su posición es el máximo de rojo menos verde, que no se
# confunde con los reflejos blancos ni con la iluminación del LED.
realce = cv2.GaussianBlur(rec_color[:, :, 2].astype(float) -
                          rec_color[:, :, 1].astype(float), (0, 0), 3)
y_laser, x_laser = np.unravel_index(np.argmax(realce), realce.shape)
if POSICION_TRAMPA is None:
    POSICION_TRAMPA = (int(x_laser), int(y_laser))
print(f"láser en ({x_laser}, {y_laser}); trampa en {POSICION_TRAMPA}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
ax[0].imshow(rec_color[:, :, ::-1])
ax[0].plot(*POSICION_TRAMPA, '+', color='cyan', ms=16, mew=1.8)
ax[0].set_xlabel('$x$ [px]')
ax[0].set_ylabel('$y$ [px]')
ax[1].imshow(realce, cmap='inferno')
ax[1].plot(x_laser, y_laser, '+', color='cyan', ms=16, mew=1.8)
ax[1].set_xlabel('$x$ [px]')
ax[1].set_ylabel('$y$ [px]')
for a in ax:
    a.grid(False)
plt.tight_layout()
guardar('campo')
plt.show()

In [ ]:
# %% 4. Frames de referencia para marcar las anclas a mano.
# De estas figuras se leen las coordenadas (x, y) de la partícula libre y
# de la atrapada en cada uno de estos segundos.
SEGUNDOS = [11, 13, 15, 17, 19]
for s in SEGUNDOS:
    k = round(s*FPS)
    fig, ax = plt.subplots(figsize=(13, 10))
    ax.imshow(extraer_canal(un_frame_color(k))[ROI[0]:ROI[1], ROI[2]:ROI[3]],
              cmap='gray')
    ax.grid(True, alpha=0.4)
    ax.set_xlabel(f'$x$ [px]   -   segundo {s}, frame {k}')
    ax.set_ylabel('$y$ [px]')
    plt.show()

In [ ]:
# %% 5. Seguimiento guiado por anclas
K_ESC = round(17*FPS)    # frame del escape, leído del video

DIAM_ANC = 25     # impar
RADIO_ANC = 22 # px de búsqueda alrededor de la guía

def seguir_con_anclas(anclas, diam=DIAM_ANC, radio=RADIO_ANC):
    a = np.array(anclas, dtype=float)
    a = a[np.argsort(a[:, 0])]
    ks = np.arange(int(a[0, 0]), int(a[-1, 0]) + 1)
    xg = np.interp(ks, a[:, 0], a[:, 1])
    yg = np.interp(ks, a[:, 0], a[:, 2])
    H, W = ROI[1] - ROI[0], ROI[3] - ROI[2]
    lado = 2*(radio + diam)
    cap = cv2.VideoCapture(VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(ks[0]))
    xs, ys, hall = [], [], 0
    for i, k in enumerate(ks):
        ok, fr = cap.read()
        if not ok:
            break
        g = extraer_canal(fr)[ROI[0]:ROI[1], ROI[2]:ROI[3]]
        x0 = int(min(max(round(xg[i]) - lado//2, 0), max(W - lado, 0)))
        y0 = int(min(max(round(yg[i]) - lado//2, 0), max(H - lado, 0)))
        rec = g[y0:y0 + lado, x0:x0 + lado]
        banda = np.abs(cv2.GaussianBlur(rec, (0, 0), 1) -
                       cv2.blur(rec, (diam, diam)))
        f = tp.locate(banda, diam, minmass=None, invert=False, engine=ENGINE)
        if len(f):
            f = f.copy()
            f['x'] += x0
            f['y'] += y0
            f['d'] = np.hypot(f['x'] - xg[i], f['y'] - yg[i])
            f = f[f['d'] < radio]
        if len(f):
            p = f.loc[f['d'].idxmin()]
            xs.append(float(p['x']))
            ys.append(float(p['y']))
            hall += 1
        else:
            xs.append(xg[i])
            ys.append(yg[i])
    cap.release()
    print(f"   {hall}/{len(ks)} frames refinados sobre la guía")
    return ks[:len(xs)], np.array(xs)*ESCALA_UM_PX, np.array(ys)*ESCALA_UM_PX

In [ ]:
# %% 6. Trayectorias, velocidad de escape y fuerza máxima
ANCLAS_LIBRE = [(248, 1200, 590), (293, 970, 590), (338, 700, 295),
                (383, 453, 295), (428, 180, 295)]
ANCLAS_ATRAPADA = [(248, 634, 538), (293, 634, 538), (383, 600, 580),
                   (428, 380, 550)]

print("libre:");     kl, xl, yl = seguir_con_anclas(ANCLAS_LIBRE)
print("atrapada:");  ka, xa_, ya_ = seguir_con_anclas(ANCLAS_ATRAPADA)

# velocidad del flujo en la ventana previa al escape
m = (kl >= K_ESC - 20) & (kl <= K_ESC)
(vx, _), cx = np.polyfit(kl[m]*DT, xl[m], 1, cov=True)
(vy, _), cy = np.polyfit(kl[m]*DT, yl[m], 1, cov=True)
v_esc = np.hypot(vx, vy)
ev = np.hypot(vx*np.sqrt(cx[0, 0]), vy*np.sqrt(cy[0, 0]))/v_esc
F = 6*np.pi*ETA_MEDIDA*1e-3*RADIO_UM*1e-6*v_esc*1e-6
rel = np.sqrt((ETA_MEDIDA_ERR/ETA_MEDIDA)**2 + (RADIO_UM_ERR/RADIO_UM)**2 +
              (ev/v_esc)**2 + (ESCALA_UM_PX_ERR/ESCALA_UM_PX)**2)
print(f"\nv_esc = {v_esc:.2f} +- {ev:.2f} um/s")
print(f"F_max = {F*1e12:.3f} +- {F*rel*1e12:.3f} pN  ({100*rel:.0f} %)")

t_esc = K_ESC*DT
fig, ax = plt.subplots(2, 1, figsize=(6.5, 5.6), sharex=True)
for a, (sl, sa, lab) in enumerate([(xl, xa_, r'$x$ [$\mu$m]'),
                                   (yl, ya_, r'$y$ [$\mu$m]')]):
    ax[a].plot(kl*DT, sl, '.', color='#d1495b', ms=3, label='libre')
    ax[a].plot(ka*DT, sa, '.', color='#00798c', ms=3, label='atrapada')
    ax[a].axvspan(t_esc - 2*DT, t_esc + 2*DT, color='0.6', alpha=0.5,
                  label='escape' if a == 0 else None)
    ax[a].set_ylabel(lab)
ax[0].legend(frameon=False, markerscale=3)
ax[1].set_xlabel('$t$ [s]')
plt.tight_layout()
guardar('arrastre_xy')
plt.show()

In [ ]:
# %% 7. Figura con el detalle de la ventana del escape
t0, t1 = (K_ESC - 20)*DT, K_ESC*DT
mm = (kl >= K_ESC - 20) & (kl <= K_ESC)
pfit = np.polyfit(kl[mm]*DT, xl[mm], 1)

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(kl*DT, xl, '.', color='maroon', ms=3, label='libre')
ax.plot(ka*DT, xa_, '.', color='forestgreen', ms=3, label='atrapada')
ax.axvspan(t0, t1, color='0.6', alpha=0.45, label='ventana del escape')
ax.set_xlabel('$t$ [s]')
ax.set_ylabel(r'$x$ [$\mu$m]')
ax.legend(frameon=False, markerscale=3, loc='upper right')

axin = ax.inset_axes([0.05, 0.06, 0.30, 0.28])
axin.plot(kl[mm]*DT, xl[mm], '.', color='maroon', ms=4)
axin.plot(kl[mm]*DT, np.polyval(pfit, kl[mm]*DT), '-', color='k', lw=1.3)
axin.set_xlim(t0, t1)
mar = 0.08*(xl[mm].max() - xl[mm].min())
axin.set_ylim(xl[mm].min() - mar, xl[mm].max() + mar)
axin.tick_params(labelsize=7)
axin.grid(alpha=0.25)
ax.indicate_inset_zoom(axin, edgecolor='0.4', lw=0.9)
plt.tight_layout()
guardar('trayectorias_inset')
plt.show()

In [ ]:
# %% 8. Control
for n in (10, 15, 20, 30, 40, 60):
    mm = (kl >= K_ESC - n) & (kl <= K_ESC)
    if mm.sum() < 5:
        continue
    (ux, _), cu = np.polyfit(kl[mm]*DT, xl[mm], 1, cov=True)
    (uy, _), cv = np.polyfit(kl[mm]*DT, yl[mm], 1, cov=True)
    v = np.hypot(ux, uy)
    e = np.hypot(ux*np.sqrt(cu[0, 0]), uy*np.sqrt(cv[0, 0]))/v
    print(f"  ventana {n:3d} frames ({n*DT:4.2f} s): v = {v:5.2f} +- {e:4.2f} um/s")

In [ ]:
# %% 9. Controles cinemáticos del escape
W_SG = 11            # impar
FRAC_ESCAPE = 0.5    # fracción de la velocidad del flujo que define el escape
T_ASINT = 1.5        # s finales que se usan como tramo asintótico

vx_at = savgol_filter(xa_, W_SG, 3, deriv=1, delta=DT)
vy_at = savgol_filter(ya_, W_SG, 3, deriv=1, delta=DT)
v_at = np.hypot(vx_at, vy_at)

supera = (v_at > FRAC_ESCAPE*v_esc).astype(int)
cand = np.flatnonzero(np.convolve(supera, np.ones(5), 'same') == 5)
if len(cand):
    print(f"escape por cinemática: frame {ka[cand[0]]}, t = {ka[cand[0]]*DT:.2f} s"
          f"   (vos pusiste {K_ESC}, t = {K_ESC*DT:.2f} s)")

m = ka*DT > ka[-1]*DT - T_ASINT
(ux, _), cux = np.polyfit(ka[m]*DT, xa_[m], 1, cov=True)
(uy, _), cuy = np.polyfit(ka[m]*DT, ya_[m], 1, cov=True)
v_lib = np.hypot(ux, uy)
ev_lib = np.hypot(ux*np.sqrt(cux[0, 0]), uy*np.sqrt(cuy[0, 0]))/v_lib
print(f"velocidad asintótica de la escapada: {v_lib:.2f} +- {ev_lib:.2f} um/s "
      f"sobre {m.sum()} frames")
print(f"velocidad de la libre:               {v_esc:.2f} um/s")

In [ ]:
# %% 10. Desplazamiento desde el centro de la trampa y rigidez efectiva
V_ESC = 11.8       
V_ESC_ERR = 0.6
T_SIN_FLUJO = 5.0       # s; tramo inicial sin arrastre
RADIO_TRAMPA_SEG = 15   # px

ka, xat, yat = seguir_con_anclas(
    [(K_INI, POSICION_TRAMPA[0], POSICION_TRAMPA[1]),
     (K_ESC, POSICION_TRAMPA[0], POSICION_TRAMPA[1])], radio=RADIO_TRAMPA_SEG)

# Centro de la trampa: donde se para cuando no hay arrastre.
m0 = ka*DT < T_SIN_FLUJO
x0, y0 = xat[m0].mean(), yat[m0].mean()
ruido = np.hypot(xat[m0].std(ddof=1), yat[m0].std(ddof=1))/np.sqrt(2)

# Desplazamiento en los 10 frames previos al escape.
m1 = (ka >= K_ESC - 10) & (ka <= K_ESC)
dr = np.hypot(xat[m1].mean() - x0, yat[m1].mean() - y0)
edr = np.hypot(np.hypot(xat[m1].std(ddof=1), yat[m1].std(ddof=1))/np.sqrt(m1.sum()),
               ruido/np.sqrt(m0.sum()))

print(f"centro de la trampa: ({x0:.3f}, {y0:.3f}) um   "
      f"promediado sobre {m0.sum()} frames")
print(f"ruido de localización: {ruido*1e3:.0f} nm")
print(f"dr_esc = {dr*1e3:.0f} +- {edr*1e3:.0f} nm")

if dr < 3*ruido:
    print("   dr_esc no supera 3 veces el ruido: k_eff no es medible")
else:
    F = 6*np.pi*ETA_MEDIDA*1e-3*RADIO_UM*1e-6*V_ESC*1e-6
    rel_F = np.sqrt((ETA_MEDIDA_ERR/ETA_MEDIDA)**2 + (RADIO_UM_ERR/RADIO_UM)**2 +
                    (V_ESC_ERR/V_ESC)**2)
    k_eff = F/(dr*1e-6)
    rel_k = np.hypot(rel_F, edr/dr)
    print(f"F_max = {F*1e12:.3f} +- {F*rel_F*1e12:.3f} pN")
    print(f"k_eff = {k_eff*1e6:.2f} +- {k_eff*rel_k*1e6:.2f} pN/um")

s = (xat - x0)*0 + np.hypot(xat - x0, yat - y0)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(ka*DT, s*1e3, '.', color='#00798c', ms=3)
ax.axhline(ruido*1e3, color='0.5', ls=':', lw=1)
ax.axvspan(0, T_SIN_FLUJO, color='0.8', alpha=0.5)
ax.axvspan((K_ESC - 10)*DT, K_ESC*DT, color='#d1495b', alpha=0.2)
ax.set_xlabel('$t$ [s]')
ax.set_ylabel(r'$|\Delta r|$ desde el centro [nm]')
plt.tight_layout()
guardar('desplazamiento_trampa')
plt.show()